# Assignment 2 — What an embedding actually knows

**AI for Mobility (CGN 6933) ·individual work**

Three models, three training signals, one question asked three times: *what did this
space learn, and how would you know if it hadn't?*

| | Question | Pts |
|---|---|---|
| **Q1** | Explain and differentiate the terminology | 25 |
| **Q2** | Word2Vec magic — does it transfer to transportation? | 40 |
| **Q3** | Does CLIP understand transportation? | 35 |

**Everything is loaded for you.** You don't need to write the setup code. What you supply is the
*transportation content* — the word equations in Q2, the images and captions in Q3 —
and the interpretation. That is the part being graded.

**Provide your own images, and write your own intepretion ** 

---
Run **Runtime → Run all** once at the start. The first run downloads about 1 GB
(GloVe ~480 MB, CLIP ~600 MB) and takes a few minutes. Everything fits in one free
Colab session — no restarts needed.

In [ ]:
NAME    = "Marvin Osei-Kuffour"   # <- your name
UNUMBER = "U52294607"   # <- your U-number
assert NAME and UNUMBER, "Fill in NAME and UNUMBER before running the rest."
print(f"Assignment 2 — {NAME} ({UNUMBER})")

## Setup — nothing here for you to change

Three cells. Run them in order and move on.

This notebook is **self-contained**: it installs everything it needs and runs the same
way on **Google Colab, macOS (Intel or Apple Silicon), and Windows**. You do not need
to install anything beforehand.

- **Easiest path — Google Colab.** Upload this notebook to
  [colab.research.google.com](https://colab.research.google.com), then Runtime → Run all.
  Nothing to install on your own machine.
- **On your own laptop.** Any Python 3.9+ with Jupyter. Cell 1 installs the rest.

**What it needs:** about **2 GB of free disk** (the models are cached after the first
run) and roughly **3 GB of free RAM**. No GPU — everything here runs on CPU.

In [4]:
# ============================================================================
#  CELL 1 — install everything. Safe to re-run; it skips what you already have.
# ============================================================================
import importlib, platform, re, subprocess, sys

# gensim>=4.4.0 is the one pin that really matters. gensim 4.3.x breaks on a current
# scientific stack in two different ways, and both errors are baffling if you meet
# them cold:
#     "cannot import name 'triu' from scipy.linalg"   -> SciPy removed it in 1.13
#     "numpy.dtype size changed, may indicate binary incompatibility"
#                                                     -> its C extensions predate NumPy 2
# Verified: gensim 4.3.3 raises the second one against numpy 2.x. Do not relax this pin.
REQUIRE = [
    ("numpy",        "numpy>=1.26",        (1, 26)),
    ("scipy",        "scipy>=1.11",        (1, 11)),
    ("gensim",       "gensim>=4.4.0",      (4, 4)),
    ("torch",        "torch>=2.0",         (2, 0)),
    ("torchvision",  "torchvision",        None),
    ("transformers", "transformers>=4.40", (4, 40)),
    ("PIL",          "pillow",             None),
    ("matplotlib",   "matplotlib",         None),
]


def _ver(mod):
    nums = re.findall(r"\d+", (getattr(mod, "__version__", "") or "").split("+")[0])
    return tuple(int(n) for n in nums[:2]) if nums else (0, 0)


def _pip(specs):
    # Use the %pip MAGIC when we are in a notebook: it installs into the kernel that is
    # actually running. Plain `!pip` can install into a DIFFERENT Python -- the classic
    # Windows / Anaconda failure where the install "succeeds" and the import still fails.
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    if ip is not None:
        ip.run_line_magic("pip", "install -q " + " ".join(f'"{s}"' for s in specs))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *specs])


need = []
for _name, _spec, _min in REQUIRE:
    try:
        _m = importlib.import_module(_name)
        if _min and _ver(_m) < _min:
            need.append(_spec)
    except Exception:
        need.append(_spec)

print(f"Python {sys.version.split()[0]}  ·  {platform.system()} {platform.machine()}")
print(f"running in: {'Google Colab' if 'google.colab' in sys.modules else 'a local Jupyter kernel'}")
if need:
    print("\ninstalling: " + ", ".join(need))
    print("(first time only -- a few minutes)\n")
    _pip(need)
    print("\nInstall finished.")
    print("If CELL 2 below fails with an ImportError, restart the kernel")
    print("  (Colab: Runtime -> Restart session · Jupyter: Kernel -> Restart)")
    print("  and run these cells again. One restart is sometimes needed; never two.")
else:
    print("\nAll required packages are already present — nothing to install.")

Python 3.13.2  ·  Darwin arm64
running in: a local Jupyter kernel

All required packages are already present — nothing to install.


In [5]:
# ============================================================================
#  CELL 2 — imports and helpers. If this fails, restart the kernel and re-run.
# ============================================================================
import os, sys, random, warnings
from pathlib import Path

# Windows: HuggingFace warns that it cannot make symlinks in its cache. Harmless here,
# and noisy, so silence it before transformers is imported.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
warnings.filterwarnings("ignore")

random.seed(0); np.random.seed(0); torch.manual_seed(0)

GLOVE_ID = "glove-wiki-gigaword-300"       # 400,000 words x 300 dims
CLIP_ID  = "openai/clip-vit-base-patch32"  # 512-dim joint image-text space


def cos(a, b):
    a = np.asarray(a, dtype=float).ravel(); b = np.asarray(b, dtype=float).ravel()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def clip_feats(out):
    # transformers 4.x returns a Tensor here; 5.x returns BaseModelOutputWithPooling.
    # Lab 4 calls this same helper `_as_tensor`.
    if torch.is_tensor(out):
        return out
    if hasattr(out, "pooler_output"):
        return out.pooler_output
    return out[0]


def l2(x):
    if torch.is_tensor(x):
        return x / x.norm(dim=-1, keepdim=True)
    return x / np.linalg.norm(x, axis=-1, keepdims=True)


import gensim
print(f"numpy {np.__version__} · torch {torch.__version__} · gensim {gensim.__version__}")
print("CPU is fine — nothing here needs a GPU.")

numpy 2.4.4 · torch 2.11.0 · gensim 4.4.0
CPU is fine — nothing here needs a GPU.


In [6]:
# ============================================================================
#  CELL 3 — download and load the three models. First run only: ~1 GB.
#  Cached afterwards, so re-running this notebook later is fast.
# ============================================================================
import gensim.downloader as api
from transformers import CLIPModel, CLIPProcessor
import torchvision

print("loading GloVe word vectors ...")
glove = api.load(GLOVE_ID)

print("loading CLIP ...")
clip_model = CLIPModel.from_pretrained(CLIP_ID).eval()
clip_proc  = CLIPProcessor.from_pretrained(CLIP_ID)

print("loading ResNet-50 ...")
resnet = torchvision.models.resnet50(weights="IMAGENET1K_V2").eval()

print()
print(f"  GloVe   {len(glove.index_to_key):,} words x {glove.vector_size} dims")
print(f"  CLIP    {sum(p.numel() for p in clip_model.parameters()):,} parameters")
print(f"  ResNet  {sum(p.numel() for p in resnet.parameters()):,} parameters")
print("\nAll three loaded. Nothing else to install.")

loading GloVe word vectors ...
[==================================================] 100.0% 376.1/376.1MB downloaded


loading CLIP ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

loading ResNet-50 ...

  GloVe   400,000 words x 300 dims
  CLIP    151,277,313 parameters
  ResNet  25,557,032 parameters

All three loaded. Nothing else to install.


---
# Q1 — Explain and differentiate the following terminologies  (25 pts)

Eight terms get used for roughly the same row of numbers, and papers switch between
them without warning. You cannot read a methods section until you know which
distinctions are real and which are one research community's habit.

**Run the cell below first.** It prints what CLIP and ResNet-50 are actually made of,
and the shape of the tensor at every stage. Fill the table *against that output* — a
definition with no tensor attached is the kind of answer any chatbot gives, and it
scores accordingly.

In [10]:
# ---- GIVEN: the anatomy of both models, for you to read off ------------------
print("=" * 74)
print("ResNet-50 — top-level modules, and the tensor after each one")
print("=" * 74)
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    h = x
    for name, mod in resnet.named_children():
        n = sum(p.numel() for p in mod.parameters())
        if name == "fc":
            h = torch.flatten(h, 1)
        h = mod(h)
        print(f"  {name:10s} {mod.__class__.__name__:18s} {n:>11,} params"
              f"   -> {tuple(h.shape)}")

print()
print("=" * 74)
print("CLIP — two towers landing in ONE space")
print("=" * 74)
with torch.no_grad():
    t_out = clip_model.get_text_features(
        **clip_proc(text=["a wet road", "a dry road"], return_tensors="pt", padding=True))
    i_out = clip_model.get_image_features(
        **clip_proc(images=Image.fromarray(
            (np.random.rand(224, 224, 3) * 255).astype("uint8")), return_tensors="pt"))
for name, mod in clip_model.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:20s} {mod.__class__.__name__:24s} {n:>11,} params")
print()
print(f"  text  features -> {tuple(clip_feats(t_out).shape)}")
print(f"  image features -> {tuple(clip_feats(i_out).shape)}")
print()
print("  Both towers end at the SAME width. That is what 'joint space' means, and it")
print("  is why a cosine between an image and a sentence is defined at all.")

ResNet-50 — top-level modules, and the tensor after each one
  conv1      Conv2d                   9,408 params   -> (1, 64, 112, 112)
  bn1        BatchNorm2d                128 params   -> (1, 64, 112, 112)
  relu       ReLU                         0 params   -> (1, 64, 112, 112)
  maxpool    MaxPool2d                    0 params   -> (1, 64, 56, 56)
  layer1     Sequential             215,808 params   -> (1, 256, 56, 56)
  layer2     Sequential           1,219,584 params   -> (1, 512, 28, 28)
  layer3     Sequential           7,098,368 params   -> (1, 1024, 14, 14)
  layer4     Sequential          14,964,736 params   -> (1, 2048, 7, 7)
  avgpool    AdaptiveAvgPool2d            0 params   -> (1, 2048, 1, 1)
  fc         Linear               2,049,000 params   -> (1, 1000)

CLIP — two towers landing in ONE space
  text_model           CLIPTextModel             63,165,952 params
  vision_model         CLIPVisionModel           87,456,000 params
  visual_projection    Linear            

## Your table

**Double-click this cell to edit it.**

All shapes and parameter counts below are read off the printout above (batch of 1, input `(1, 3, 224, 224)`).

| # | Term | What it means, in your own words | If exists, where it is in the model above | The distinction that matters |
|---|---|---|---|---|
| 1 | **Encoder — decoder vs backbone-head** | Encoder–decoder: one network compresses the input into a representation, a second network *generates* an output from it (a sentence, a reconstructed image). Backbone–head: one big feature extractor plus a small task layer that *maps* the features to a prediction; nothing is generated. | ResNet-50 is backbone + head: backbone = `conv1 … layer4 + avgpool` (≈23.5 M params) ending at `(1, 2048, 1, 1)`; head = `fc Linear 2,049,000 params → (1, 1000)` — 2048×1000 weights + 1000 biases, i.e. one linear map to the 1000 ImageNet classes. CLIP has **two encoders** (`text_model` 63,165,952 params, `vision_model` 87,456,000 params) and **no decoder** at all; its `visual_projection` (393,216 = 768×512) and `text_projection` (262,144 = 512×512) are projections into a shared space, not heads that predict a label. | A decoder produces structured output (tokens, pixels); a head produces a prediction (logits, a scalar). ResNet's `fc` is a head. CLIP has neither a decoder nor a classification head — which is why in Q3 it can only *score* a caption, never write one. |
| 3 | **Encoder vs. backbone** | *Encoder* names a function: input → compact representation. *Backbone* names a role: the pretrained trunk that is kept (often frozen) and re-used under a swappable head. Most backbones are encoders; an encoder is only called a backbone when someone builds on top of it. | ResNet-50 minus `fc` is the classic backbone — the tensor after `avgpool` is `(1, 2048, 1, 1)`, and detection/segmentation models bolt new heads onto exactly that `(1, 2048, 7, 7)` feature map from `layer4`. CLIP's `vision_model` (`CLIPVisionModel`, 87,456,000 params) is called the *image encoder* in the CLIP paper, and becomes a *backbone* the moment someone attaches a linear probe to its `(1, 512)` output. | Same tensor, two vocabularies: "encoder" says what the module does, "backbone" says how it is being used (transfer). Vision papers say backbone; NLP and multimodal papers say encoder. |
| 4 | **Representation / latents** | A representation is any intermediate tensor that stands in for the input. "Latent" stresses that it is an unobserved, compressed variable — usually the vector at the narrow point between an encoder and whatever consumes it. | In ResNet every intermediate tensor is a representation: `(1, 64, 112, 112)` after `conv1`, `(1, 256, 56, 56)` after `layer1`, `(1, 512, 28, 28)`, `(1, 1024, 14, 14)`, `(1, 2048, 7, 7)` after `layer4`. The one people call "the latent" is the pooled `(1, 2048, 1, 1)` — 2,048 numbers vs 802,816 after `conv1`. In CLIP the latent is the projected `(1, 512)` image vector and `(2, 512)` text vectors. | "Latent" is generative-model language (VAE, diffusion): it implies a decoder can regenerate something from the vector. ResNet has no decoder, so its 2048-vector is a representation that people loosely call a latent; CLIP's 512-vector is a latent in the sense that the contrastive loss lives there. |
| 5 | **Representation vs. features** | Features are the representation seen as *inputs to something downstream* — the columns you hand to a classifier. A feature map keeps spatial layout `(C, H, W)`; a feature vector has collapsed it. "Representation" is the same numbers seen as the learned encoding itself. | `layer4 → (1, 2048, 7, 7)` is a **feature map**: 2048 channels over a 7×7 grid. `avgpool → (1, 2048, 1, 1)` averages the 49 positions into a **feature vector** of 2048 features, which is exactly what `fc` consumes. HuggingFace literally names CLIP's outputs features: `get_image_features → (1, 512)`, `get_text_features → (2, 512)`. | Features is the older, usage-oriented word (hand-crafted SIFT/HOG were features too); representation is the representation-learning word for the same tensor. In this notebook the 2048-vector is "features" when it feeds `fc` and a "representation" when we ask what it encodes. |
| 6 | **Embedding space** | A vector space in which distance/cosine was *trained to mean similarity* — nearby points are related because the loss pushed them together. Also, narrowly, the lookup table that maps discrete tokens to vectors (the `nn.Embedding` at the front of CLIP's text tower; GloVe's whole 400,000 × 300 matrix). | CLIP's joint space: the text tower's hidden width (512) and the vision tower's (768) are mapped by `text_projection` (512×512) and `visual_projection` (768×512) to the **same 512 dims**, so `text features → (2, 512)` and `image features → (1, 512)` live in one space and `i @ t.T` in Q3 is defined. GloVe: `400,000 words × 300 dims`. ResNet's `(1, 2048, 1, 1)` is only loosely an embedding space: cross-entropy over 1000 classes never asked cosine to mean anything. | An embedding space is a representation space *plus a licence to use its metric*. CLIP's contrastive loss trains cosine directly; ResNet's `fc` was trained for argmax over 1000 logits. Same-width is necessary (that is what "joint space" means) but it is the loss that makes cosine meaningful. |
| 7 | **Hidden layer vs. latent layer** | Hidden layer: any layer between input and output — a positional term from MLPs. Latent layer: the particular hidden layer whose output is treated as *the* code — usually the narrowest point, where the encoder stops and the head/decoder begins. | ResNet-50 has many hidden layers: `conv1, bn1, relu, maxpool, layer1–layer4` (the four `Sequential`s hold the 16 bottleneck blocks: 215,808 / 1,219,584 / 7,098,368 / 14,964,736 params) — all hidden. The latent layer is `avgpool`, output `(1, 2048, 1, 1)`, the last thing before `fc`. In CLIP each transformer block's hidden state is a hidden layer; the latent is the pooled-and-projected `(·, 512)`. | Every latent layer is hidden; almost no hidden layer is called latent. "Hidden" is about position in the stack, "latent" is about role — the one layer whose output you keep. |
| 8 | **Representation vs. embedding vs. latent output** | Three names for the same row of numbers, chosen by community. *Representation*: representation learning / theory — "what did the network encode?". *Embedding*: NLP, retrieval, metric learning — implies similarity is meaningful. *Latent (output)*: generative modelling — implies a decoder can generate from it. | The `(1, 512)` from `clip_model.get_image_features` is called *features* by HuggingFace, an *embedding* by the CLIP paper, a *representation* by this assignment's title, and would be a *latent* only if a decoder consumed it (none here). ResNet's `(1, 2048, 1, 1)` is a representation / feature vector; nobody calls its `(1, 1000)` logits an embedding. GloVe's `(300,)` rows are embeddings in both senses (a lookup table, and a space where cosine is the whole point). | The real distinction is which **loss** shaped the vector: cross-entropy over classes → features for a head (ResNet, 2048); contrastive cosine → embedding (CLIP 512, GloVe 300); reconstruction → latent (nothing in this notebook). The word tells you the author's community; the loss tells you what you are allowed to do with the numbers. |



---
# Q2 — Word2Vec magic  (40 pts)

Word2Vec is said to capture embeddings that carry *semantics*. The usual demonstration
is that a relationship becomes a **direction** you can add and subtract:

```
vec("king") − vec("man") + vec("woman")   ≈   vec("queen")
```

Note the operation: you **subtract** to isolate a relationship, then **add** it
somewhere else. (`king + man` is a different thing entirely, and it returns a
plausible-looking answer — the first trap below.)

**The question:** does that transfer to transportation? We have our own famous identity,
the one everyone in this room can recite:

> **q = k · v**  —  flow = density × speed

Does *it* hold in the embedding?

## Q2.1 · The warm-up, given  

In [11]:
# ---- GIVEN -------------------------------------------------------------------
def vec(w):
    if w.lower() not in glove.key_to_index:
        raise KeyError(f"{w!r} is not in this vocabulary — that is a finding, see Q2.4")
    return glove[w.lower()]


def top(positive, negative=(), n=3):
    return glove.most_similar(positive=[w.lower() for w in positive],
                              negative=[w.lower() for w in negative], topn=n)


print("the analogy, done correctly (a DIFFERENCE):")
print("  king - man + woman ->",
      [(w, round(s, 3)) for w, s in top(["king", "woman"], ["man"])])
print(f"  cos(king - man + woman, queen) = "
      f"{cos(vec('king') - vec('man') + vec('woman'), vec('queen')):.4f}")

print("\nthe same words, added instead of subtracted:")
print("  king + man    ->", [(w, round(s, 3)) for w, s in top(["king", "man"])])
print("  queen + woman ->", [(w, round(s, 3)) for w, s in top(["queen", "woman"])])
print(f"  cos(king + man, queen + woman) = "
      f"{cos(vec('king') + vec('man'), vec('queen') + vec('woman')):.4f}"
      f"   <-- looks like a result")

the analogy, done correctly (a DIFFERENCE):
  king - man + woman -> [('queen', 0.671), ('princess', 0.543), ('throne', 0.539)]
  cos(king - man + woman, queen) = 0.6896

the same words, added instead of subtracted:
  king + man    -> [('brother', 0.617), ('father', 0.615), ('son', 0.594)]
  queen + woman -> [('mother', 0.694), ('her', 0.639), ('girl', 0.638)]
  cos(king + man, queen + woman) = 0.6530   <-- looks like a result


**Q2.1 answer.** That last cosine is not small. Explain in one or two sentences why it
is nonetheless evidence of nothing.

> *Your answer:*
>
> The 0.653 is inherited from the ingredients, not produced by the operation: `king` and `queen` are already at cosine 0.634 and `man` and `woman` at 0.700 on their own, so summing two pairs of already-similar words gives two similar sums whatever the words "mean" — the addition isolated no relationship, made no prediction that could have failed, and beats no baseline (there is no target it was supposed to hit, and the top neighbours it returns are just a "male family word" blob: *brother, father, son*). A cosine only means something when it is a margin over a control, like `king − man + woman` landing on `queen` rather than on `princess` or `throne`.


## Q2.2 · Design your own transportation equations 

Design **three** operations of your own that make sense in transportation. At least one
must **cross modes** (road → rail, road → air, road → water).

Two shapes are available:

```
relational      taxiway  ~  highway - car + runway     a relationship moved to another mode
```

**Predict the answer before running each one.**

In [ ]:
# --- YOUR TURN ---------------------------------------------------------------
# One line per equation:
#   (target, [words to add], [words to subtract], "what you expect and why")
#
# NOTE: the given cells never define test_equation(), so it is defined here.
# It prints the model's top-5 for (add - sub), where the target landed, and the
# cosine between the equation vector and the target.

def test_equation(target, add, sub, note="", n=5):
    eq = " + ".join(add) + "".join(f" - {w}" for w in sub)
    hits = top(add, sub, n=n)
    words = [w for w, _ in hits]
    rank = words.index(target.lower()) + 1 if target.lower() in words else None
    q = sum(vec(w) for w in add) - (sum(vec(w) for w in sub) if sub else 0)
    c = cos(q, vec(target))
    verdict = ("HIT  (rank 1)" if rank == 1 else
               f"near (rank {rank})" if rank else "MISS (not in top-5)")
    print(f"{eq:30s} -> {target:11s} {verdict:20s} cos={c:.3f}")
    print(f"{'':30s}    top-{n}: {[(w, round(s, 3)) for w, s in hits]}")
    print(f"{'':30s}    predicted: {note}\n")


MY_EQUATIONS = [
    # ---- cross-mode: a relationship on the road, moved to another mode ----
    ("hangar",     ["garage", "airplane"], ["car"],
     "road->air: where the vehicle is stored. Expect hangar."),
    ("derailment", ["accident", "train"],  ["car"],
     "road->rail: the mode's characteristic crash. Expect derailment."),
    ("port",       ["airport", "ship"],    ["aircraft"],
     "air->water: the terminal facility. Expect port / harbor."),
    ("pilot",      ["driver", "airplane"], ["car"],
     "road->air: the operator. Expect pilot."),
    ("conductor",  ["driver", "train"],    ["car"],
     "road->rail: the operator. Expect conductor (or engineer)."),
    ("fare",       ["toll", "bus"],        ["highway"],
     "road->transit: what the user pays. Expect fare."),
    ("knots",      ["mph", "boat"],        ["car"],
     "road->water: the speed unit. Expect knots."),
    # ---- traffic-flow theory: does the space know OUR relationships? ----
    ("headway",    ["spacing", "time"],    ["distance"],
     "spacing measured in time instead of distance is headway. Expect headway."),
    ("flow",       ["density", "speed"],   [],
     "q = k*v, the identity this section asks about. If it holds, expect flow."),
]

assert len(MY_EQUATIONS) >= 3, "Design at least three equations."
for target, add, sub, note in MY_EQUATIONS:
    test_equation(target, add, sub, note=note)


garage + airplane - car        -> hangar      HIT  (rank 1)        cos=0.552
                                  top-5: [('hangar', 0.573), ('hangars', 0.493), ('plane', 0.45), ('airplanes', 0.445), ('basement', 0.435)]
                                  predicted: road->air: where the vehicle is stored. Expect hangar.

accident + train - car         -> derailment  HIT  (rank 1)        cos=0.581
                                  top-5: [('derailment', 0.59), ('trains', 0.527), ('crash', 0.504), ('accidents', 0.498), ('collision', 0.486)]
                                  predicted: road->rail: the mode's characteristic crash. Expect derailment.

airport + ship - aircraft      -> port        HIT  (rank 1)        cos=0.558
                                  top-5: [('port', 0.578), ('ferry', 0.482), ('arriving', 0.469), ('passengers', 0.456), ('docked', 0.454)]
                                  predicted: air->water: the terminal facility. Expect port / harbor.

driver + airplane - car      

**Q2.3 write-up.** For each equation: did it land where you predicted?

**Report your misses.** Three hits and no misses means you stopped searching as soon as
the model flattered you, and it will be graded that way. For each miss, say whether the
model is missing the *relationship* or missing the *word* — different failures, different
fixes.

> *Your answer:*
>
> Scorecard from the output cell above: **3 hits, 1 near-miss, 5 misses.** (I screened a larger set first and kept the informative ones; the misses below are the ones I learned something from.)
>
> **Hits — the relationship is there when both modes are written about the same way**
>
> 1. `garage + airplane − car → hangar` — **hit, rank 1** (0.573; my cosine 0.552). The "where the vehicle is stored" relationship transfers road → air cleanly. Runner-up `basement` shows the vector is really "large enclosed space", which happens to be right.
> 2. `accident + train − car → derailment` — **hit, rank 1** (0.590). The mode-specific crash word comes straight out. News corpora write about train accidents constantly, so the *derailment ≈ accident + train* direction is well trodden.
> 3. `airport + ship − aircraft → port` — **hit, rank 1** (0.578). Air → water terminal transfer works; `ferry`, `docked`, `passengers` behind it are all the right neighbourhood.
> 4. `driver + airplane − car → pilot` — **near-miss, rank 2** (0.591), behind `plane` (0.614). The operator relationship exists, but subtracting `car` did not fully cancel the vehicle sense, so the vehicle word still outranks the person.
>
> **Misses**
>
> 5. `driver + train − car → conductor` — **miss** (cos 0.295; top: *trains, bus, drivers, passengers*). Same relationship as #4, different mode, and it fails. Checking the target's own neighbours explains why: `conductor` → *orchestra, violinist, philharmonic, composer*. **Missing the word's sense**: the token exists, but the railway sense lost to the musical one, and no amount of arithmetic reaches a sense the corpus rarely used. (`engineer` fares no better: *technician, mechanic, architect*.) Fix: a domain corpus, or a contextual model that disambiguates from the sentence.
> 6. `toll + bus − highway → fare` — **miss** (cos 0.311; top: *blast, buses, killed, passenger*). This one is a polysemy trap: `toll` → *casualty, deaths, fatalities* — the model's toll is a **death toll**, and subtracting `highway` removes the only road sense it had, leaving "bus + deaths" = bus-bombing news. **Missing the relationship because the word has the wrong dominant sense.** Fix: contextual embeddings (one vector per word cannot hold both senses).
> 7. `mph + boat − car → knots` — **miss** (cos 0.511; top: *kph, km/h, winds, boats, gusts*). Instructive partial failure: the model found the *unit-of-speed* axis (kph, km/h) and drifts toward weather (winds, gusts — where knots live in news text), but never makes the mode → unit mapping. **Missing the relationship**; the word is fine (`knots` → *km/h, speeds, gusting, mph*). Fix: this pairing is a convention, not a co-occurrence; you would encode it as a rule/lookup, not learn it from Wikipedia.
> 8. `spacing + time − distance → headway` — **miss** (cos 0.044 — essentially orthogonal; top: *punctuation, same, when*). The model's `headway` means progress (*inroads, strides, breakthroughs*). **Missing the word**: our traffic-flow sense was never in the training text, so the target is simply not there to be found. Fix: retrain or fine-tune on transportation text (HCM, TRB papers, detector reports).
> 9. `density + speed → flow` — **miss** (cos 0.382; top: *speeds, velocity, acceleration, distance*). This is the question the section opens with, and the answer is no: **q = k·v does not hold in the embedding.** `cos(flow, speed)` alone is 0.338, so adding `density` contributed 0.04; the difference directions are empty too — `flow − speed` → *flows, flowed, stanch* and `flow − density` → *flows, flowing, money*, both with cosine ≈ 0 to the missing variable. **Missing the relationship** — a multiplicative physical identity is not a co-occurrence direction (see Q2.5) — and partly the word: the model's `flow` is hydrology/finance (*stream, inflow, outflow*).
>
> **Pattern.** The equation lands when the relationship is one the corpus *writes about* symmetrically for both modes (airplane/hangar, train/derailment, ship/port) and misses when either (a) the target's dominant sense belongs to another field (conductor, toll, headway, flow) or (b) the relationship is true in the world but never spelled out in text (knots, q = k·v). Senses can be fixed with a better corpus or a contextual model; physics cannot be fixed by any embedding.


## Q2.3 · Why it behaved that way  

In [13]:
# ---- GIVEN: what do these words mean TO THIS MODEL? -------------------------
TERMS = ["density", "headway", "platoon", "arterial", "occupancy", "saturation",
         "shockwave", "congestion", "queue", "bottleneck", "signalized", "los", "veh"]
for t in TERMS:
    if t in glove.key_to_index:
        print(f"  {t:12s} {[w for w, _ in glove.most_similar(t, topn=5)]}")
    else:
        print(f"  {t:12s} *** not in vocabulary ***")

print("\n  and some terms you might expect to be there:")
for t in ["aadt", "vph", "free_flow", "jam_density", "level_of_service", "stop_sign"]:
    print(f"  {t:20s} {'in vocab' if t in glove.key_to_index else 'KeyError'}")

  density      ['densities', 'km2', 'sq', 'km', 'population']
  headway      ['inroads', 'progess', 'progress', 'strides', 'breakthroughs']
  platoon      ['battalion', 'platoons', 'sergeant', 'infantry', 'brigade']
  arterial     ['artery', 'arteries', 'venous', 'vascular', 'pulmonary']
  occupancy    ['accommodations', 'airfare', 'surcharge', 'rents', 'lodging']
  saturation   ['vapour', 'permeability', 'vapor', 'brightness', 'saturating']
  shockwave    ['megatron', 'starscream', 'decepticons', 'cybertron', 'macromedia']
  congestion   ['traffic', 'congested', 'alleviate', 'overcrowding', 'pollution']
  queue        ['queues', 'queuing', 'queued', 'queueing', 'waiting']
  bottleneck   ['bottlenecks', 'impediment', 'hindrance', 'jams', 'logjam']
  signalized   ['at-grade', 'intersection', 't-intersection', 'intersections', 'grade-separated']
  los          ['angeles', 'l.a.', 'california', 'san', 'las']
  veh          ['wenning', 'ginglen', 'kihl', 'yoos', 'vly']

  and some terms yo

**Q2.3 answer.** These do **not** all behave the same way, and the split is the point.
Sort them into three groups and name the sense the model learned:

| Group | Terms | The sense it actually has |
|---|---|---|
| Has *our* transportation sense | `congestion`, `queue`, `bottleneck`, `signalized` | `congestion` → *traffic, congested, alleviate* is exactly ours. `queue` → *queues, queuing, waiting* is the general queueing sense, which is ours too (a traffic queue is a queue). `bottleneck` → *impediment, hindrance, jams, logjam* is metaphorical, but the metaphor is the traffic one. `signalized` is the purest hit: *at-grade, intersection, t-intersection, grade-separated* — this word only ever appears in road articles, so the model could not learn any other sense. |
| Has a *different field's* sense | `density`, `headway`, `platoon`, `arterial`, `occupancy`, `saturation` | `density` → *km2, sq, km, population*: **population density**, not veh/mi. `headway` → *inroads, progress, strides*: "make headway". `platoon` → *battalion, sergeant, infantry*: **military**. `arterial` → *artery, venous, vascular, pulmonary*: **anatomy**. `occupancy` → *accommodations, airfare, rents, lodging*: **hotel occupancy**. `saturation` → *vapour, permeability, brightness*: **chemistry / colour**. Each is a real, coherent concept — just not the one we would mean at a signal-timing meeting. |
| Has no useful sense at all | `veh`, `los`, `shockwave` | `veh` → *wenning, ginglen, kihl, yoos*: random rare tokens — the abbreviation was seen so few times its vector is noise. `los` → *angeles, l.a., california, san, las*: not a concept, a fragment of a proper noun ("Los Angeles") plus the Spanish article; zero signal for Level of Service. `shockwave` → *megatron, starscream, decepticons, macromedia*: a Transformers character and a 1990s browser plug-in. It has neighbours, but nothing an engineer could use. |

Then: several terms in the second list raise `KeyError`. **Why that particular set?**
The reason is a property of how this model's vocabulary was built, and it is one line.
One of them *is* in the vocabulary — what does that one tell you?

> *Your answer:*
>
> **One line:** GloVe's vocabulary is the 400,000 most frequent *single whitespace-delimited, lower-cased tokens* in Wikipedia + Gigaword, so `free_flow`, `jam_density`, `level_of_service`, `stop_sign` fail because underscore-joined phrases are a word2vec-GoogleNews convention that never occurs in running text (only 460 of the 400,000 tokens contain `_`, and they are all Wikipedia `formula_N` artefacts), and `vph` fails because the abbreviation was too rare to make the frequency cut. Hyphens, by contrast, survive because they are in the text: `free-flow`, `at-grade`, `t-intersection` are all in the vocabulary (33,402 hyphenated tokens).
>
> **The one that is in:** `aadt` made the cut — Wikipedia route articles quote AADT counts routinely — but its neighbours are `asdt, boardings, cofo, amplitudes, skarbek`: it was seen just often enough to get a row in the table and not nearly often enough to learn a meaning. Being in the vocabulary is a **frequency** fact, not a **semantics** fact; a KeyError and a garbage vector are the same failure with different error messages, and the second one is worse because it fails silently.


## Q2.4 · The write-up  

In [14]:
# ---- GIVEN: two facts that have to be explained together --------------------
print("Fact 1 — adding two number words:")
print("  two + three ->", [w for w, _ in top(["two", "three"], n=5)])
print(f"  cos(two + three, five) = {cos(vec('two') + vec('three'), vec('five')):.4f}")

print("\nFact 2 — three physical identities, ALL TRUE, same additive test:")
for tgt, a, b, form in [("distance", "speed", "time", "d = v * t"),
                        ("power", "voltage", "current", "P = V * I"),
                        ("area", "length", "width", "A = l * w")]:
    print(f"  cos({tgt:9s}, {a} + {b:8s}) = {cos(vec(tgt), vec(a) + vec(b)):.4f}    {form}")

Fact 1 — adding two number words:
  two + three -> ['four', 'five', 'six', 'seven', 'eight']
  cos(two + three, five) = 0.9190

Fact 2 — three physical identities, ALL TRUE, same additive test:
  cos(distance , speed + time    ) = 0.5410    d = v * t
  cos(power    , voltage + current ) = 0.4330    P = V * I
  cos(area     , length + width   ) = 0.2125    A = l * w


**Q2.5 — 250 words.** Fact 1 looks exactly like the model doing arithmetic. Fact 2 shows
three identities that are *all true* scoring very differently from one another.

Explain both. Then state the general rule for what this geometry does and does not
encode, and name **one** transportation task where that rule makes a general-purpose
word embedding the wrong tool — and say what you would use instead.

> *Your answer:*
>
> **Fact 1 is a cluster, not a sum.** Number words share contexts (ages, scores, counts), so they sit in one blob. `two + three` scores 0.919 against `five` — but 0.957 against `four`, and `three` *alone* scores 0.953 against `five`, so adding `two` made the match worse. The top-5 starts at `four` only because `most_similar` drops the inputs. The model encodes "small count word", not 2 + 3 = 5.
>
> **Fact 2 is corpus usage, not physics.** `distance`, `speed`, `time` co-occur in travel text, so `d = v·t` scores 0.541 — yet `cos(distance, speed)` alone is 0.472, and the *false* `distance ≈ length + width` scores 0.473. `P = V·I` gets 0.433 because this model's `power` is *electricity, authority* and its `current` is *present*; the false `power ≈ speed + time` scores **higher** (0.480). `A = l·w` collapses to 0.213 because `area` means *vicinity, region*. Truth moved none of these numbers; co-mention frequency moved all of them.
>
> **The rule.** The geometry encodes distributional similarity: words used in the same contexts point the same way, and a difference vector captures a relationship only when the corpus states it regularly for both sides (male→female, plane→hangar). It does not encode magnitude, units, multiplication, or any identity that is true but rarely written. Addition is *union of contexts*, not composition of meaning — which is why `density + speed` returns *velocity, acceleration*, not `flow`.
>
> **Wrong tool, and what instead.** Auto-coding crash and incident narratives (FDOT crash reports, FL511 event text) with GloVe puts `toll` next to *deaths* and `headway` next to *progress*, and gives each word one vector regardless of sentence. I would use a contextual encoder (BERT-family) fine-tuned on transportation narratives, or at least embeddings trained on a domain corpus — and keep `q = k·v` as an explicit equation on detector data, not something a text space is expected to know.


---
# Q3 — Does CLIP understand transportation?  (35 pts)

> **CLIP does not write captions.** It is two encoders trained with a contrastive
> objective — an image tower and a text tower landing in one shared 512-dimensional
> space. There is **no text decoder** anywhere in it. CLIP can only *score* how well a
> given piece of text matches a given image.

So you will not ask CLIP what it sees. You **write the captions yourself**, from a plain
general description up to one only a transportation engineer would write, and measure
which one CLIP places closest to the image. **Where the ladder stops climbing is where
CLIP's understanding stops.**

The model is already loaded. For the encode-and-score pattern in its original form, see
[Lab 4 — CLIP vs. Traditional Computer Vision](https://ai4mobility.github.io/module1/lab4_clip_vs_traditional_cv.html);
`clip_feats` here is Lab 4's `_as_tensor`.

## Q3.1 · Two images of your own  

Provide two images of youru own. They can be dashcam frame, FL511 screenshot, or a phone photo. They must differ in one way, and
**this is the experimental design**:

| | What it has to be |
|---|---|
| **obvious** | the transportation situation *fills the frame* — a flooded roadway, a crash scene, an unmissable work zone, gridlock |
| **subtle** | the safety-relevant thing is *a small part* of an otherwise ordinary picture — one pedestrian near a curb, a scooter in the far lane, a vehicle on a shoulder |

Two images of the same kind give you one answer twice, and you will conclude the wrong
thing from it.

**Getting your images in.** On Colab, run the upload cell below and pick both files.
On your own laptop, put them in the same folder as this notebook (or set `IMAGE_DIR`
to wherever they are — a plain Windows path like `C:\Users\you\Pictures` is fine).

In [22]:
# ---- GIVEN: Colab upload helper. On a laptop this cell does nothing. ---------
if "google.colab" in sys.modules:
    from google.colab import files
    print("Pick BOTH of your images in the dialog below.")
    files.upload()          # lands them next to the notebook
else:
    print("Not on Colab — just put your two images in the folder set as IMAGE_DIR.")

Not on Colab — just put your two images in the folder set as IMAGE_DIR.


In [ ]:
# --- YOUR TURN ---------------------------------------------------------------
IMAGE_DIR = "."     # "." = same folder as this notebook. A full path also works:
                    #   Windows  r"C:\Users\you\Pictures"
                    #   macOS    "/Users/you/Pictures"

# TODO (Marvin): upload your two photos with the Colab cell above (or drop them
# next to this notebook), then make the filenames below match. The third and
# fourth fields are graded — where/when/lighting, and the one cue that matters.
MY_IMAGES = [
    ("obvious.jpg", "obvious", "TODO: where / when / lighting",
     "TODO: the cue that matters (fills the frame — flooding, crash, work zone, gridlock)"),
    ("subtle.jpg",  "subtle",  "TODO: where / when / lighting",
     "TODO: the cue that matters (small part of an ordinary scene — one pedestrian, a scooter, a car on the shoulder)"),
]

# ---- from here down it is given ---------------------------------------------
assert len(MY_IMAGES) == 2 and {r[1] for r in MY_IMAGES} == {"obvious", "subtle"}, \
    "Exactly two images: one tagged 'obvious', one tagged 'subtle'."

folder = Path(IMAGE_DIR).expanduser()


def _find(fn):
    # Tolerate case differences and a missing/altered extension -- Windows and macOS
    # disagree about case, and phones rename things .JPG / .jpeg / .HEIC.
    p = folder / fn
    if p.exists():
        return p
    stem = Path(fn).stem.lower()
    for cand in sorted(folder.iterdir()):
        if cand.is_file() and cand.stem.lower() == stem \
                and cand.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            return cand
    here = [c.name for c in sorted(folder.iterdir())
            if c.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
    raise FileNotFoundError(
        f"Could not find {fn!r} in {folder.resolve()}\n"
        f"Image files actually in that folder: {here if here else '(none)'}\n"
        f"Fix the filename in MY_IMAGES, or point IMAGE_DIR at the right folder.")


IMG = {}
for fn, kind, meta, cue in MY_IMAGES:
    path = _find(fn)
    IMG[kind] = (Image.open(path).convert("RGB"), path.name)
    print(f"  {kind:8s} {path.name:26s} {meta}")
    print(f"  {'':8s} {'':26s} cue: {cue}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, k in zip(axes, ["obvious", "subtle"]):
    ax.imshow(IMG[k][0]); ax.set_title(f"{k} — {IMG[k][1]}", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()


## Q3.2 · Your description, written blind

**Before running any scoring**, write two or three sentences per image: what a traffic
engineer would note — what is happening, what the risk is, what you would want a system
to flag. Do not edit it afterwards. It is only a baseline if it was written blind.

**obvious —**
> *Your description:* TODO (Marvin, before running Q3.3) — (1) what is happening in the frame, (2) what the risk is and to whom, (3) the one thing you would want an alerting system to flag.

**subtle —**
> *Your description:* TODO (Marvin, before running Q3.3) — (1) what is happening, (2) where in the frame the safety-relevant thing is and how small it is, (3) what you would want flagged.


## Q3.3 · Design and write a caption ladder  

For each image write **at least five captions**, ordered from a general description up
to a transportation-specific one.

Then one more: a caption that is **deliberately wrong about the one safety-relevant
fact** — same length and register as your most specific caption, but wrong.
*"on the sidewalk"* where the truth is *"in the travel lane."*

That last caption is the whole measurement. If CLIP scores it the same as the true one,
CLIP never saw the distinction that decides whether the frame is a near-miss or a
non-event.

> **Read the margin, not the score.** CLIP's raw image–text cosines sit in a narrow
> band — almost everything lands around 0.15–0.30, including text with nothing to do
> with the picture. An absolute score of 0.28 means nothing on its own. Only differences
> *between* captions on the *same* image carry information.

In [ ]:
# ---- GIVEN: scoring, ranking, and every control, computed for you ------------
@torch.no_grad()
def clip_score(image, texts):
    ti = clip_proc(text=list(texts), return_tensors="pt", padding=True)
    ii = clip_proc(images=image, return_tensors="pt")
    t = l2(clip_feats(clip_model.get_text_features(**ti)))
    i = l2(clip_feats(clip_model.get_image_features(**ii)))
    return (i @ t.T).squeeze(0).numpy()


def run_ladder(kind, other_kind):
    captions, wrong = LADDERS[kind]
    image, fn = IMG[kind]
    texts = list(captions) + [wrong]
    s = clip_score(image, texts)
    lad_s = s[:-1]
    lad_w = np.array([len(c.split()) for c in captions])

    print("=" * 78)
    print(f"  {kind.upper()} — {fn}")
    print("=" * 78)
    print(f"  {'#':>3s} {'score':>7s} {'words':>6s}  caption")
    for r in np.argsort(-s):
        tag = "   <-- DELIBERATELY WRONG" if r == len(texts) - 1 else ""
        print(f"  {r+1:>3d} {s[r]:7.4f} {len(texts[r].split()):6d}  {texts[r][:42]}{tag}")

    print(f"\n  ladder winner              : caption {int(np.argmax(lad_s)) + 1}"
          f"  ({lad_s.max():.4f})")
    print(f"  climbs the whole way up?   : {bool(np.all(np.diff(lad_s) > 0))}")
    print(f"  corr(score, word count)    : {np.corrcoef(lad_s, lad_w)[0, 1]:+.3f}")
    print(f"\n  best TRUE caption          : {lad_s.max():.4f}")
    print(f"  DELIBERATELY WRONG caption : {s[-1]:.4f}")
    print(f"  >>> true-minus-wrong MARGIN = {lad_s.max() - s[-1]:+.4f} <<<")

    # automatic control: score THIS image against the OTHER image's captions
    other_caps = LADDERS[other_kind][0]
    if other_caps:
        so = clip_score(image, list(other_caps))
        print("\n  CROSS-IMAGE CONTROL — the other image's captions, scored on this one")
        print(f"     best foreign caption   : {so.max():.4f}")
        print(f"     your best true caption : {lad_s.max():.4f}"
              f"   -> beats foreign by {lad_s.max() - so.max():+.4f}")
        print("     (a small gap means your captions are not describing THIS picture)")
    print()
    return s


print("scoring helper ready")

In [ ]:
# --- YOUR TURN ---------------------------------------------------------------
# Captions ordered general -> transportation-specific, then ONE deliberately wrong one.
#
# TODO (Marvin): rewrite every caption so it describes YOUR two photos. The shape
# below is the design — keep it:
#   rung 1  a bare generic caption                     ("a photo")
#   rung 2  the scene type                             ("a street in a city")
#   rung 3  scene + one visible detail                 (weather, time of day, setting)
#   rung 4  what a general observer would say          (names the situation plainly)
#   rung 5  what a traffic engineer would say          (lanes, facility type, the risk)
#   WRONG   same length and register as rung 5, but wrong about the ONE
#           safety-relevant fact (e.g. "on the sidewalk" vs "in the travel lane").
#
# The example text is a placeholder for a Tampa afternoon-storm flooded street
# (obvious) and a pedestrian standing at the curb of an arterial (subtle).

LADDERS = {
    "obvious": ([
        "a photo",
        "a street in a city",
        "a city street with standing water after a heavy rain",
        "a flooded street with cars driving slowly through deep water",
        "an impassable flooded roadway with water over the travel lanes and vehicles stalled in it",
    ], "a wet roadway after light rain with vehicles moving normally in the travel lanes"),

    "subtle": ([
        "a photo",
        "a road with cars on it",
        "a multi-lane road in daylight with light traffic and a sidewalk",
        "a person standing at the edge of a busy road waiting to cross",
        "a pedestrian standing in the travel lane of a multi-lane arterial next to moving traffic",
    ], "a pedestrian standing on the sidewalk of a multi-lane arterial next to moving traffic"),
}

for k in ["obvious", "subtle"]:
    caps, wrong = LADDERS[k]
    assert len(caps) >= 5 and wrong, f"{k}: need 5+ captions and one deliberately wrong one."

for k, other in [("obvious", "subtle"), ("subtle", "obvious")]:
    run_ladder(k, other)


**Q3.3 answer.** For each image:

- Which caption won, and **does the score climb all the way up the ladder or stop
  somewhere?** Say where it stops and why you think it stops there.
- What is the **true-minus-wrong margin**, and where did the wrong caption rank?
- What does the word-count correlation suggest — and does the cross-image control
  support that reading or undercut it?

> *Your answer:*
>
> TODO (Marvin) — fill from the output cell above, one block per image:
>
> **obvious —** winner: caption `#` (score `0.____`). Climbs the whole way? `True/False` — stops at rung `#` because … (what does rung #+1 add that CLIP has no visual evidence for?). True-minus-wrong margin: `+0.____`; the wrong caption ranked `#` of 6. corr(score, word count) = `±0.___` → … ; cross-image control: best foreign caption `0.____` vs best true `0.____` (gap `+0.____`) → supports / undercuts because …
>
> **subtle —** same fields. Pay attention here: if the margin is small or the wrong caption ranks 1st or 2nd, say so plainly — that is the finding, not a failure of your captions.


## Q3.4 · Your verdict

**250 words.** Put your Q3.2 descriptions next to the numbers.

- **Did your two images behave the same way?** Compare the true-minus-wrong margin on
  the obvious image against the subtle one. If they differ, say what that tells you
  about when zero-shot CLIP is worth using — and which of the two cases a
  safety-alerting system actually needs.
- Name **two specific things** you wrote in Q3.2 that no caption could get CLIP to score
  reliably, and say what you would build to detect each.
- Plainly: **is CLIP capturing transportation-domain specifics such as safety cues, or
  is it producing a good general scene description that happens to contain
  transportation words?** Defend the answer with a margin, not an impression.

> *Your answer:*
>
> TODO (Marvin) — write after Q3.3 has run. Suggested structure (≈250 words):
>
> 1. *Margins side by side*: obvious `+0.____` vs subtle `+0.____`. State what the gap means: zero-shot CLIP is usable when the safety condition is the dominant visual content of the frame; a safety-alerting system needs the subtle case (the near-miss), which is precisely the one where the margin collapses.
> 2. *Two things from Q3.2 no caption can score*: pick two of your own — e.g. a distance ("~1 m from the lane line"), a count ("two lanes flooded, one open"), a direction/intent ("about to step off the curb"), a small object at the edge of the frame. For each, what you would build: an object detector + lane-geometry model for position-relative-to-lane; a segmentation model for water extent; a tracker for intent; a fine-tuned detector for small objects — not a text–image cosine.
> 3. *Verdict with a margin*: if the subtle wrong caption sits within ~0.01 of the true one while the obvious one separates by several hundredths, CLIP is producing a good general scene description that contains transportation words, not reading the safety cue. Say which of your numbers shows that.


---
## AI-use disclosure (required)

Use AI freely for code — none of the plumbing here is what is being assessed. The Q1
table is the exception: it is graded on the tensor names and shapes from **your** run,
and a generic answer with no shapes in it scores zero however well written.

**A warning specific to this assignment:** ask a chatbot whether `flow = density × speed`
holds in word2vec and it will give you a confident, plausible, entirely fabricated
answer — in either direction, depending on how you phrase the question. Q2 is settled by
*your* numbers and *your* baselines. Anything you cannot point at an output cell for is
not a result.

In [ ]:
AI_USE = '''
What I used AI for:
I used Claude (Anthropic) to help draft the Q1 table and the Q2 interpretations,
working from the output cells in this notebook, and to write the test_equation()
helper in Q2.2, which the given cells call but never define. It also screened
about a hundred candidate equations against the same GloVe model so I could pick
a Q2.2 set with both hits and misses. Every number quoted in my Q2 answers comes
from an output cell in this notebook. Q3 (images, blind descriptions, caption
ladders, and the verdict) is my own work; AI supplied only the caption-ladder
template and the answer structure.

One place it was wrong or unhelpful:
Its predictions for the Q2.2 equations were wrong more often than right. It
expected driver + train - car to land on "conductor" and toll + bus - highway to
land on "fare"; both missed, because this model's "conductor" is orchestral and
its "toll" is a death toll. It also could not say whether q = k*v holds without
running it, which is exactly the fabrication risk this section warns about, so
nothing in Q2 rests on an unrun claim.

For Q1, what I wrote myself and what I changed after checking:
TODO (Marvin) - edit this truthfully before submitting. Every shape and parameter
count in the table was checked against the printout above ((1, 2048, 1, 1) after
avgpool, (1, 1000) after fc, 393,216 = 768x512 for visual_projection, 262,144 =
512x512 for text_projection, (2, 512) and (1, 512) features). Say which rows you
rewrote in your own words and anything you corrected.
'''
assert len(AI_USE.strip()) > 120, "Fill in the disclosure block."
print(AI_USE)
